## Detect Depression

### MiniMax-M2.5

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.siliconflow.cn/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="Pro/MiniMaxAI/MiniMax-M2.5",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/MiniMax-M2.5_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 1408 条样本


第1轮: 100%|██████████| 1408/1408 [2:30:31<00:00,  6.41s/it]  

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [1]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/MiniMax-M2.5_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true_original = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 将y_true中的2和3合并到1
y_true = [0 if label == 0 else 1 for label in y_true_original]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 86.15%
Precision: 66.26%
Recall: 58.76%
F1 Score: 62.28%


### MiniMax-M2.7

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="MiniMax-M2.7",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=4000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/MiniMax-M2.7_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(6)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [1]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/MiniMax-M2.7_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true_original = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 将y_true中的2和3合并到1
y_true = [0 if label == 0 else 1 for label in y_true_original]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 85.72%
Precision: 65.67%
Recall: 55.84%
F1 Score: 60.36%


### Pro/zai-org/GLM-5

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.siliconflow.cn/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="Pro/zai-org/GLM-5",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=4000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/GLM-5_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [11]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/GLM-5_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 86.93%
Precision: 65.85%
Recall: 68.25%
F1 Score: 67.03%


### stepfun-ai/Step-3.5-Flash

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.siliconflow.cn/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="stepfun-ai/Step-3.5-Flash",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=4000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/Step-3.5-Flash_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 22 条样本


第1轮: 100%|██████████| 22/22 [03:24<00:00,  9.31s/it]

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [8]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/Step-3.5-Flash_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 86.86%
Precision: 65.61%
Recall: 68.25%
F1 Score: 66.91%


### baidu/ERNIE-4.5-300B-A47B

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api2.aigcbest.top/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="baidu/ernie-4.5-300b-a47b",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/ERNIE-4.5-300B-A47B_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 1408 条样本


第1轮: 100%|██████████| 1408/1408 [1:01:32<00:00,  2.62s/it]

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [6]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/ERNIE-4.5-300B-A47B_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 83.38%
Precision: 54.95%
Recall: 81.02%
F1 Score: 65.49%


### deepseek-ai/DeepSeek-V3.2

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.siliconflow.cn/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3.2",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/DeepSeek-V3.2_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 1408 条样本


第1轮:   0%|          | 0/1408 [00:00<?, ?it/s]

第1轮: 100%|██████████| 1408/1408 [1:19:48<00:00,  3.40s/it]

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [3]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/DeepSeek-V3.2_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 82.46%
Precision: 53.09%
Recall: 84.67%
F1 Score: 65.26%


### Native Qwen3.5-0.8B-Base

In [1]:
import json
import csv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 文件路径
test_csv_path = "/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv"
pred_jsonl_path = "/data/zhengtianlong/Private/Depressed/LlamaFactory/saves/Qwen3.5-0.8B-Base/lora/Native_eval_26-03-18-02-45-11/generated_predictions.jsonl"

def map_label(predict_str):
    """根据预测字符串提取标签：如果包含 '0' 返回0，否则如果包含 '1' 返回1，否则默认返回1"""
    if "0" in predict_str:
        return 0
    elif "1" in predict_str:
        return 1
    else:
        return 1  # 默认返回1

# 1. 读取测试集标签
true_labels = []
with open(test_csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)  # 跳过表头
    for row in reader:
        if len(row) < 3:
            continue
        target = row[2]  # 第三列是 target
        # target 可能为字符串 '0' 或 '1'，转为整数
        true_labels.append(int(target))

# 2. 读取预测结果
pred_labels = []
with open(pred_jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        predict = data.get("predict", "")
        pred_labels.append(map_label(predict))

# 3. 检查数量一致性
if len(true_labels) != len(pred_labels):
    print(f"警告：真实标签数量 {len(true_labels)}，预测标签数量 {len(pred_labels)}，将按较小数量计算。")
    min_len = min(len(true_labels), len(pred_labels))
    true_labels = true_labels[:min_len]
    pred_labels = pred_labels[:min_len]

# 4. 计算指标
accuracy = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, pos_label=1) # pos_label=1 指定正类为抑郁（1）
recall = recall_score(true_labels, pred_labels, pos_label=1)
f1 = f1_score(true_labels, pred_labels, pos_label=1)

# 5. 打印结果
print("评估结果：")
print(f"准确率 (Accuracy): {accuracy*100:.2f}")
print(f"精确率 (Precision): {precision*100:.2f}")
print(f"召回率 (Recall): {recall*100:.2f}")
print(f"F1 值 (F1-score): {f1*100:.2f}")

评估结果：
准确率 (Accuracy): 48.01
精确率 (Precision): 22.14
召回率 (Recall): 66.42
F1 值 (F1-score): 33.21


### Native Qwen3.5-2B-Base

In [2]:
import json
import csv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 文件路径
test_csv_path = "/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv"
pred_jsonl_path = "/data/zhengtianlong/Private/Depressed/LlamaFactory/saves/Qwen3.5-2B-Base/lora/Native_eval_26-03-18-02-45-11/generated_predictions.jsonl"

def map_label(predict_str):
    """根据预测字符串提取标签：如果包含 '0' 返回0，否则如果包含 '1' 返回1，否则默认返回1"""
    if "0" in predict_str:
        return 0
    elif "1" in predict_str:
        return 1
    else:
        return 1  # 默认返回1

# 1. 读取测试集标签
true_labels = []
with open(test_csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)  # 跳过表头
    for row in reader:
        if len(row) < 3:
            continue
        target = row[2]  # 第三列是 target
        # target 可能为字符串 '0' 或 '1'，转为整数
        true_labels.append(int(target))

# 2. 读取预测结果
pred_labels = []
with open(pred_jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        predict = data.get("predict", "")
        pred_labels.append(map_label(predict))

# 3. 检查数量一致性
if len(true_labels) != len(pred_labels):
    print(f"警告：真实标签数量 {len(true_labels)}，预测标签数量 {len(pred_labels)}，将按较小数量计算。")
    min_len = min(len(true_labels), len(pred_labels))
    true_labels = true_labels[:min_len]
    pred_labels = pred_labels[:min_len]

# 4. 计算指标
accuracy = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, pos_label=1) # pos_label=1 指定正类为抑郁（1）
recall = recall_score(true_labels, pred_labels, pos_label=1)
f1 = f1_score(true_labels, pred_labels, pos_label=1)

# 5. 打印结果
print("评估结果：")
print(f"准确率 (Accuracy): {accuracy*100:.2f}")
print(f"精确率 (Precision): {precision*100:.2f}")
print(f"召回率 (Recall): {recall*100:.2f}")
print(f"F1 值 (F1-score): {f1*100:.2f}")

评估结果：
准确率 (Accuracy): 70.38
精确率 (Precision): 35.79
召回率 (Recall): 65.69
F1 值 (F1-score): 46.33


### Native Qwen3.5-4B-Base

In [3]:
import json
import csv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 文件路径
test_csv_path = "/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv"
pred_jsonl_path = "/data/zhengtianlong/Private/Depressed/LlamaFactory/saves/Qwen3.5-4B-Base/lora/Native_eval_26-03-18-02-45-11/generated_predictions.jsonl"

def map_label(predict_str):
    """根据预测字符串提取标签：如果包含 '0' 返回0，否则如果包含 '1' 返回1，否则默认返回1"""
    if "0" in predict_str:
        return 0
    elif "1" in predict_str:
        return 1
    else:
        return 1  # 默认返回1

# 1. 读取测试集标签
true_labels = []
with open(test_csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)  # 跳过表头
    for row in reader:
        if len(row) < 3:
            continue
        target = row[2]  # 第三列是 target
        # target 可能为字符串 '0' 或 '1'，转为整数
        true_labels.append(int(target))

# 2. 读取预测结果
pred_labels = []
with open(pred_jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        predict = data.get("predict", "")
        pred_labels.append(map_label(predict))

# 3. 检查数量一致性
if len(true_labels) != len(pred_labels):
    print(f"警告：真实标签数量 {len(true_labels)}，预测标签数量 {len(pred_labels)}，将按较小数量计算。")
    min_len = min(len(true_labels), len(pred_labels))
    true_labels = true_labels[:min_len]
    pred_labels = pred_labels[:min_len]

# 4. 计算指标
accuracy = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, pos_label=1) # pos_label=1 指定正类为抑郁（1）
recall = recall_score(true_labels, pred_labels, pos_label=1)
f1 = f1_score(true_labels, pred_labels, pos_label=1)

# 5. 打印结果
print("评估结果：")
print(f"准确率 (Accuracy): {accuracy*100:.2f}")
print(f"精确率 (Precision): {precision*100:.2f}")
print(f"召回率 (Recall): {recall*100:.2f}")
print(f"F1 值 (F1-score): {f1*100:.2f}")

评估结果：
准确率 (Accuracy): 73.15
精确率 (Precision): 38.60
召回率 (Recall): 64.23
F1 值 (F1-score): 48.22


### Native Qwen3.5-9B-Base

In [4]:
import json
import csv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 文件路径
test_csv_path = "/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv"
pred_jsonl_path = "/data/zhengtianlong/Private/Depressed/LlamaFactory/saves/Qwen3.5-9B-Base/lora/Native_eval_26-03-18-02-45-11/generated_predictions.jsonl"

def map_label(predict_str):
    """根据预测字符串提取标签：如果包含 '0' 返回0，否则如果包含 '1' 返回1，否则默认返回1"""
    if "0" in predict_str:
        return 0
    elif "1" in predict_str:
        return 1
    else:
        return 1  # 默认返回1

# 1. 读取测试集标签
true_labels = []
with open(test_csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)  # 跳过表头
    for row in reader:
        if len(row) < 3:
            continue
        target = row[2]  # 第三列是 target
        # target 可能为字符串 '0' 或 '1'，转为整数
        true_labels.append(int(target))

# 2. 读取预测结果
pred_labels = []
with open(pred_jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        predict = data.get("predict", "")
        pred_labels.append(map_label(predict))

# 3. 检查数量一致性
if len(true_labels) != len(pred_labels):
    print(f"警告：真实标签数量 {len(true_labels)}，预测标签数量 {len(pred_labels)}，将按较小数量计算。")
    min_len = min(len(true_labels), len(pred_labels))
    true_labels = true_labels[:min_len]
    pred_labels = pred_labels[:min_len]

# 4. 计算指标
accuracy = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, pos_label=1) # pos_label=1 指定正类为抑郁（1）
recall = recall_score(true_labels, pred_labels, pos_label=1)
f1 = f1_score(true_labels, pred_labels, pos_label=1)

# 5. 打印结果
print("评估结果：")
print(f"准确率 (Accuracy): {accuracy*100:.2f}")
print(f"精确率 (Precision): {precision*100:.2f}")
print(f"召回率 (Recall): {recall*100:.2f}")
print(f"F1 值 (F1-score): {f1*100:.2f}")

评估结果：
准确率 (Accuracy): 75.71
精确率 (Precision): 41.90
召回率 (Recall): 64.23
F1 值 (F1-score): 50.72


### Native Qwen3.5-35B-A3B-Base

In [1]:
import json
import csv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 文件路径
test_csv_path = "/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv"
pred_jsonl_path = "/data/zhengtianlong/Private/Depressed/LlamaFactory/saves/Qwen3.5-35B-A3B-Base/lora/Native_eval_26-03-18-02-45-11/generated_predictions.jsonl"

def map_label(predict_str):
    """根据预测字符串提取标签：如果包含 '0' 返回0，否则如果包含 '1' 返回1，否则默认返回1"""
    if "0" in predict_str:
        return 0
    elif "1" in predict_str:
        return 1
    else:
        return 1  # 默认返回1

# 1. 读取测试集标签
true_labels = []
with open(test_csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)  # 跳过表头
    for row in reader:
        if len(row) < 3:
            continue
        target = row[2]  # 第三列是 target
        # target 可能为字符串 '0' 或 '1'，转为整数
        true_labels.append(int(target))

# 2. 读取预测结果
pred_labels = []
with open(pred_jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        predict = data.get("predict", "")
        pred_labels.append(map_label(predict))

# 3. 检查数量一致性
if len(true_labels) != len(pred_labels):
    print(f"警告：真实标签数量 {len(true_labels)}，预测标签数量 {len(pred_labels)}，将按较小数量计算。")
    min_len = min(len(true_labels), len(pred_labels))
    true_labels = true_labels[:min_len]
    pred_labels = pred_labels[:min_len]

# 4. 计算指标
accuracy = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, pos_label=1) # pos_label=1 指定正类为抑郁（1）
recall = recall_score(true_labels, pred_labels, pos_label=1)
f1 = f1_score(true_labels, pred_labels, pos_label=1)

# 5. 打印结果
print("评估结果：")
print(f"准确率 (Accuracy): {accuracy*100:.2f}")
print(f"精确率 (Precision): {precision*100:.2f}")
print(f"召回率 (Recall): {recall*100:.2f}")
print(f"F1 值 (F1-score): {f1*100:.2f}")

评估结果：
准确率 (Accuracy): 77.98
精确率 (Precision): 44.74
召回率 (Recall): 55.84
F1 值 (F1-score): 49.68
